Let's first verify that a GPU is indeed connected and recognized by the system.

In [9]:
from torch.nn.modules import activation
import os
import torch
import pandas as pd
from tqdm import tqdm
from google.colab import drive
drive.mount("/content/drive")

from tqdm.auto import tqdm

model_name = "google/gemma-2-2b-it"
import torch.nn as nn
import torch.nn.functional as F
moo = "Gemma-2-2b-it"
BASE_DIR = f"/content/drive/MyDrive/{model_name}_Activations"
CSV_PATH = os.path.join(BASE_DIR, f"Reward_bench_activations/layer 24/Gemma-2-2b-it_reward_hacking_rescored.csv")
Syco_CSV = pd.read_csv(f"/content/drive/MyDrive/{model_name}_Activations/{moo}_sycophant_rescored")
df = pd.read_csv(CSV_PATH)
activation_path_reward_hacking = os.path.join(BASE_DIR,f"Reward_bench_activations")

print("Rows:", len(df))
print(df["reward_hacked"].value_counts())
print(Syco_CSV["sycophantic"].value_counts())
class SAE(nn.Module):
    def __init__(self, d_in, d_hidden, l1_coeff=3e-4):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_hidden, bias=False)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        self.l1_coeff = l1_coeff

        # Tied weights option (recommended for interpretability)
        self.decoder.weight = nn.Parameter(self.encoder.weight.T)

    def forward(self, x):
        # Sparse latent features
        z = F.relu(self.encoder(x))
        # Reconstruction
        x_hat = self.decoder(z)
        return x_hat, z

    def loss(self, x, x_hat, z):
        mse = ((x - x_hat)**2).mean()
        l1 = self.l1_coeff * z.abs().sum(dim=-1).mean()
        return mse + l1


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 2985
reward_hacked
False    1870
True     1115
Name: count, dtype: int64
sycophantic
False    6691
True     5051
Name: count, dtype: int64


In [10]:
import shutil
import os
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────
DRIVE_RB_DIR  = activation_path_reward_hacking  # source on Drive
LOCAL_RB_DIR  = "/content/reward_bench_activations"  # temp local copy
os.makedirs(LOCAL_RB_DIR, exist_ok=True)

# ── Get list of files to copy ──────────────────────────────────────────
all_files = [f for f in os.listdir(DRIVE_RB_DIR) if f.endswith(".pt")]
print(f"Found {len(all_files)} activation files to copy.")

# ── Copy to local disk ─────────────────────────────────────────────────
copied  = 0
skipped = 0

for fname in tqdm(all_files):
    src = os.path.join(DRIVE_RB_DIR, fname)
    dst = os.path.join(LOCAL_RB_DIR, fname)

    if os.path.exists(dst):
        skipped += 1
        continue

    shutil.copy2(src, dst)
    copied += 1

print(f"Copied:  {copied}")
print(f"Skipped: {skipped} (already present)")
print(f"Total:   {len(all_files)} files ready at {LOCAL_RB_DIR}")

Found 2985 activation files to copy.


100%|██████████| 2985/2985 [00:00<00:00, 196861.50it/s]

Copied:  0
Skipped: 2985 (already present)
Total:   2985 files ready at /content/reward_bench_activations


In [11]:
hacked_acts = []
clean_acts  = []

# Start iteration from the second row (index 1) to skip the header which is treated as data.
# Access 'run_id' via column index 0 and 'reward_hacked' via column index 5.
for _, row in tqdm(df.iloc[1:].iterrows(), total=len(df) - 1):
    run_id = int(row[0]) # Access run_id by numeric index
    hacked = (row["reward_hacked"]) == True  # Access reward_hacked by numeric index and convert to boolean

    path = os.path.join(LOCAL_RB_DIR, f"run_{run_id:05d}.pt")
    #path = os.path.join(f"/content/drive/MyDrive/SAE_Works_Pythia-12B/run_{run_id:05d}.pt"
    if not os.path.exists(path):
        continue

    sample = torch.load(path)

    acts_list = sample["activations"]  # list of [1, seq, d_model]

    acts = torch.cat(acts_list, dim=1)       # [1, seq, d_model]
    acts = acts.reshape(-1, acts.shape[-1])  # [tokens, d_model]
    acts = acts.float()                      # fp32 for SAE

    if hacked:
        hacked_acts.append(acts)
    else:
        clean_acts.append(acts)

X_hacked = torch.cat(hacked_acts, dim=0)
X_clean  = torch.cat(clean_acts, dim=0)

print("Hacked tokens:", X_hacked.shape)
print("Clean tokens :", X_clean.shape)



  0%|          | 0/2984 [00:00<?, ?it/s]/tmp/ipykernel_39527/222310264.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  run_id = int(row[0]) # Access run_id by numeric index
100%|██████████| 2984/2984 [00:54<00:00, 54.48it/s]


Hacked tokens: torch.Size([214434, 2304])
Clean tokens : torch.Size([354171, 2304])


In [12]:
print(f"X_hacked shape: {X_hacked.shape}")
print(f"X_clean shape:  {X_clean.shape}")
print(f"X_hacked size:  {X_hacked.element_size() * X_hacked.nelement() / 1e9:.2f} GB")
print(f"X_clean size:   {X_clean.element_size() * X_clean.nelement() / 1e9:.2f} GB")

X_hacked shape: torch.Size([214434, 2304])
X_clean shape:  torch.Size([354171, 2304])
X_hacked size:  1.98 GB
X_clean size:   3.26 GB


In [13]:
base_dir = "/content/drive/MyDrive/"

In [14]:
base_dir = "/content/drive/MyDrive/"
def save_in_chunks(tensor, path, chunk_size=500_000):
    """Save a large tensor in chunks atomically to avoid corruption."""
    os.makedirs(path, exist_ok=True)
    n_chunks = (len(tensor) + chunk_size - 1) // chunk_size

    for i in range(n_chunks):
        chunk     = tensor[i * chunk_size : (i + 1) * chunk_size]
        final_path = os.path.join(path, f"chunk_{i:04d}.pt")
        tmp_path   = final_path + ".tmp"

        # Write to temp file first
        torch.save(chunk, tmp_path)

        # Atomic rename — if this succeeds the file is guaranteed complete
        os.replace(tmp_path, final_path)

        # Verify immediately after saving
        try:
            torch.load(final_path)
            print(f"  ✓ Chunk {i+1}/{n_chunks} saved and verified")
        except Exception as e:
            print(f"  ✗ Chunk {i+1}/{n_chunks} corrupted — retrying")
            os.remove(final_path)
            torch.save(chunk, tmp_path)
            os.replace(tmp_path, final_path)

        del chunk

# Save
print("hi")
save_in_chunks(X_hacked, os.path.join(base_dir, "X_hacked_chunks"))
print("whee")
save_in_chunks(X_clean,  os.path.join(base_dir, "X_clean_chunks"))
print("g")
#Reload later
def load_in_chunks(path):
    """Reload chunked tensor."""
    chunks = sorted([f for f in os.listdir(path) if f.startswith("chunk_")])
    return torch.cat([
        torch.load(os.path.join(path, f)) for f in chunks
    ], dim=0)
X_hacked = load_in_chunks(os.path.join(base_dir, "X_hacked_chunks"))
X_clean  = load_in_chunks(os.path.join(base_dir, "X_clean_chunks"))
#save_in_chunks(X_clean,  os.path.join(base_dir, "X_clean_syco_chunks"))
'''X_hacked = load_in_chunks(os.path.join(base_dir, "X_hacked_chunks"))
X_clean  = load_in_chunks(os.path.join(base_dir, "X_clean_chunks"))
X_clean_syco = load_in_chunks(os.path.join(base_dir, "X_clean_syco_chunks"))
X_hacked_syco = load_in_chunks(os.path.join(base_dir, "X_hacked_syco_chunks"))'''

hi
  ✓ Chunk 1/1 saved and verified
whee
  ✓ Chunk 1/1 saved and verified
g


'X_hacked = load_in_chunks(os.path.join(base_dir, "X_hacked_chunks"))\nX_clean  = load_in_chunks(os.path.join(base_dir, "X_clean_chunks"))\nX_clean_syco = load_in_chunks(os.path.join(base_dir, "X_clean_syco_chunks"))\nX_hacked_syco = load_in_chunks(os.path.join(base_dir, "X_hacked_syco_chunks"))'

In [15]:
import shutil
import os
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────
DRIVE_ACT_DIR  = os.path.join(BASE_DIR, "sycophant_activations")  # source on Drive
LOCAL_ACT_DIR  = "/content/sycophant_activations"                  # temp local copy
os.makedirs(LOCAL_ACT_DIR, exist_ok=True)

# ── Get list of files to copy ──────────────────────────────────────────
all_files = [f for f in os.listdir(DRIVE_ACT_DIR) if f.endswith(".pt")]
print(f"Found {len(all_files)} activation files to copy.")

# ── Copy to local disk ─────────────────────────────────────────────────
# Skips files already copied so rerunning this cell is safe
copied  = 0
skipped = 0

for fname in tqdm(all_files):
    src = os.path.join(DRIVE_ACT_DIR, fname)
    dst = os.path.join(LOCAL_ACT_DIR, fname)

    if os.path.exists(dst):
        skipped += 1
        continue

    shutil.copy2(src, dst)
    copied += 1

print(f"Copied:  {copied}")
print(f"Skipped: {skipped} (already present)")
print(f"Total:   {len(all_files)} files ready at {LOCAL_ACT_DIR}")

Found 13634 activation files to copy.


100%|██████████| 13634/13634 [00:00<00:00, 171323.79it/s]

Copied:  0
Skipped: 13634 (already present)
Total:   13634 files ready at /content/sycophant_activations


In [16]:
hacked_acts_syco = []
clean_acts_syco  = []
SUBSETS = ["answer", "are_you_sure", "feedback", "mimicry"]
sycophant_dir = LOCAL_ACT_DIR

# Template: {subset}_{global_idx:05d}.pt
# e.g. answer_00000.pt, are_you_sure_00001.pt, feedback_00002.pt

for _, row in tqdm(Syco_CSV.iterrows(), total=len(Syco_CSV)):
    global_idx = int(row["global_idx"])
    subset     = row["subset"]
    hacked     = row["sycophantic"] == True

    # Build path using subset name and global_idx
    path = os.path.join(sycophant_dir, f"{subset}_{global_idx:05d}.pt")

    if not os.path.exists(path):
        continue

    sample = torch.load(path)

    # Sycophancy activations are stored as a tensor directly, not a dict
    # Handle both formats just in case
    if isinstance(sample, dict):
        acts = sample["activations"]
        if isinstance(acts, list):
            acts = torch.cat(acts, dim=1)
            acts = acts.reshape(-1, acts.shape[-1])
        else:
            acts = acts.reshape(-1, acts.shape[-1])
    else:
        # Direct tensor — shape is (num_tokens, hidden_dim)
        acts = sample.reshape(-1, sample.shape[-1])

    acts = acts.float()

    if hacked:
        hacked_acts_syco.append(acts)
    else:
        clean_acts_syco.append(acts)

X_hacked_syco = torch.cat(hacked_acts_syco, dim=0)
X_clean_syco  = torch.cat(clean_acts_syco, dim=0)

save_in_chunks(X_hacked_syco, os.path.join(base_dir, "X_hacked_syco_chunks"))
print("whee")
print("bree")
save_in_chunks(X_clean_syco,  os.path.join(base_dir, "X_clean_syco_chunks"))
print("Sycophantic tokens:", X_hacked_syco.shape)
print("Clean tokens:      ", X_clean_syco.shape)

100%|██████████| 11742/11742 [00:32<00:00, 363.60it/s]


  ✓ Chunk 1/1 saved and verified
whee
bree
  ✓ Chunk 1/2 saved and verified
  ✓ Chunk 2/2 saved and verified
Sycophantic tokens: torch.Size([480982, 2304])
Clean tokens:       torch.Size([602047, 2304])


In [17]:
save_in_chunks(X_hacked, os.path.join(base_dir, "X_hacked_chunks"))
print("whee")
save_in_chunks(X_clean,  os.path.join(base_dir, "X_clean_chunks"))
save_in_chunks(X_hacked_syco, os.path.join(base_dir, "X_hacked_syco_chunks"))
save_in_chunks(X_clean_syco,  os.path.join(base_dir, "X_clean_syco_chunks"))

  ✓ Chunk 1/1 saved and verified
whee
  ✓ Chunk 1/1 saved and verified
  ✓ Chunk 1/1 saved and verified
  ✓ Chunk 1/2 saved and verified
  ✓ Chunk 2/2 saved and verified


In [18]:
import torch
import pandas as pd

def load_chunks_lazy(path):
    """Returns sorted list of chunk paths without loading them."""
    chunks = sorted([
        os.path.join(path, f)
        for f in os.listdir(path)
        if f.startswith("chunk_")
    ])
    return chunks

def rank_sae_neurons_chunked(
    sae,
    hacked_chunk_paths,
    clean_chunk_paths,
    sae_level,
    min_freq=0.05,
    batch_size=512,
    checkpoint_dir=None,
    checkpoint_interval=100,
):
    """
    Same as rank_sae_neurons but loads one chunk at a time
    instead of the full tensor — avoids RAM spike.
    """
    sae.eval()
    sae.cuda()
    d_hidden = sae.encoder.out_features

    sum_z_h        = torch.zeros(d_hidden, dtype=torch.float32)
    sum_sq_z_h     = torch.zeros(d_hidden, dtype=torch.float32)
    count_active_h = torch.zeros(d_hidden, dtype=torch.float32)
    total_tokens_h = 0

    # Process hacked chunks one at a time
    for chunk_path in tqdm(hacked_chunk_paths, desc=f"Hacked chunks (SAE {sae_level})"):
        chunk = torch.load(chunk_path)  # load one chunk

        for i in range(0, chunk.shape[0], batch_size):
            batch_X_h = chunk[i:i+batch_size].cuda()
            with torch.no_grad():
                batch_Z_h = torch.relu(sae.encoder(batch_X_h)).cpu()

            sum_z_h        += batch_Z_h.sum(dim=0)
            sum_sq_z_h     += (batch_Z_h ** 2).sum(dim=0)
            count_active_h += (batch_Z_h > 0).float().sum(dim=0)
            total_tokens_h += batch_Z_h.shape[0]

            del batch_X_h, batch_Z_h
            torch.cuda.empty_cache()

        del chunk  # free chunk immediately after processing
        gc.collect()

    sum_z_c        = torch.zeros(d_hidden, dtype=torch.float32)
    count_active_c = torch.zeros(d_hidden, dtype=torch.float32)
    total_tokens_c = 0

    # Process clean chunks one at a time
    for chunk_path in tqdm(clean_chunk_paths, desc=f"Clean chunks (SAE {sae_level})"):
        chunk = torch.load(chunk_path)

        for i in range(0, chunk.shape[0], batch_size):
            batch_X_c = chunk[i:i+batch_size].cuda()
            with torch.no_grad():
                batch_Z_c = torch.relu(sae.encoder(batch_X_c)).cpu()

            sum_z_c        += batch_Z_c.sum(dim=0)
            count_active_c += (batch_Z_c > 0).float().sum(dim=0)
            total_tokens_c += batch_Z_c.shape[0]

            del batch_X_c, batch_Z_c
            torch.cuda.empty_cache()

        del chunk
        gc.collect()

    # Compute final statistics — same as before
    mean_h = sum_z_h / total_tokens_h
    mean_c = sum_z_c / total_tokens_c

    var_h = (sum_sq_z_h / total_tokens_h) - (mean_h ** 2)
    var_h[var_h < 0] = 0

    freq_h    = count_active_h / total_tokens_h
    freq_c    = count_active_c / total_tokens_c
    delta     = mean_h - mean_c
    freq_diff = freq_h - freq_c
    score     = (delta * freq_diff) / (torch.sqrt(var_h) + 1e-6)

    df = pd.DataFrame({
        "sae_level":       sae_level,
        "neuron_id":       range(d_hidden),
        "mean_hacked":     mean_h.numpy(),
        "mean_clean":      mean_c.numpy(),
        "freq_hacked":     freq_h.numpy(),
        "freq_clean":      freq_c.numpy(),
        "variance_hacked": var_h.numpy(),
        "delta":           delta.numpy(),
        "freq_diff":       freq_diff.numpy(),
        "score":           score.numpy(),
    })

    df = df[(df["freq_hacked"] >= min_freq) & (df["delta"] > 0)]
    df = df.sort_values("score", ascending=False)
    return df

In [19]:
from scipy.stats import pointbiserialr
def compute_pointbiserial(
    hacked_chunk_paths,
    clean_chunk_paths,
    sae,
    batch_size=512,
    checkpoint_dir=None,
    checkpoint_interval=500,
):
    ckpt_path = None
    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)
        ckpt_path = os.path.join(checkpoint_dir, "pointbiserial_ckpt.json")

    def save_ckpt(correlations, last_feature_idx):
        if ckpt_path:
            tmp = ckpt_path + ".tmp"
            with open(tmp, "w") as f:
                json.dump({
                    "last_feature_idx": last_feature_idx,
                    "correlations":     correlations
                }, f)
            os.replace(tmp, ckpt_path)

    def load_ckpt():
        if ckpt_path and os.path.exists(ckpt_path):
            with open(ckpt_path) as f:
                data = json.load(f)
            print(f"  Resuming pointbiserial from feature {data['last_feature_idx']}")
            return data["correlations"], data["last_feature_idx"] + 1
        return [], 0

    # ── Get d_hidden from SAE ──────────────────────────────────────────
    d_hidden = sae.encoder.out_features

    # ── Streaming accumulators per feature ────────────────────────────
    # We compute correlation incrementally without storing full matrix
    # Using Welford-style online computation:
    # sum_x, sum_y, sum_xy, sum_x2, sum_y2, n
    print("Computing streaming accumulators...")

    n          = 0
    sum_x      = torch.zeros(d_hidden)  # sum of labels (0/1)
    sum_y      = torch.zeros(d_hidden)  # sum of feature activations
    sum_xy     = torch.zeros(d_hidden)  # sum of label * activation
    sum_y2     = torch.zeros(d_hidden)  # sum of activation^2

    def process_chunks(chunk_paths, label):
        nonlocal n, sum_x, sum_y, sum_xy, sum_y2

        for chunk_path in tqdm(chunk_paths, desc=f"{'Hacked' if label==1 else 'Clean'} chunks"):
            chunk = torch.load(chunk_path)

            for i in range(0, chunk.shape[0], batch_size):
                batch = chunk[i:i+batch_size].cuda()
                with torch.no_grad():
                    z = torch.relu(sae.encoder(batch)).cpu()

                b = z.shape[0]
                l = torch.full((b,), label, dtype=torch.float32)

                sum_x  += l.sum() * torch.ones(d_hidden)
                sum_y  += z.sum(dim=0)
                sum_xy += (l.unsqueeze(1) * z).sum(dim=0)
                sum_y2 += (z ** 2).sum(dim=0)
                n      += b

                del batch, z, l
                torch.cuda.empty_cache()

            del chunk
            gc.collect()

    process_chunks(hacked_chunk_paths, label=1)
    process_chunks(clean_chunk_paths,  label=0)

    # ── Compute point-biserial correlation from accumulators ──────────
    # r = (n*sum_xy - sum_x*sum_y) /
    #     sqrt((n*sum_x2 - sum_x^2) * (n*sum_y2 - sum_y^2))
    # where sum_x2 = sum of label^2 = sum_x (since labels are 0/1)
    print("Computing correlations from accumulators...")

    sum_x2 = sum_x  # label^2 == label for binary labels

    numerator   = n * sum_xy - sum_x * sum_y
    denom_left  = n * sum_x2 - sum_x ** 2
    denom_right = n * sum_y2 - sum_y ** 2
    denominator = torch.sqrt(denom_left * denom_right + 1e-10)

    r = numerator / denominator
    r = torch.clamp(r, -1.0, 1.0)  # numerical safety

    # ── Load checkpoint and build output ──────────────────────────────
    correlations, start_feature = load_ckpt()

    if start_feature == 0:
        # Fresh run — build from accumulators
        correlations = [
            (int(idx), float(r[idx]), 0.0)  # p-value approximated
            for idx in range(d_hidden)
        ]

    save_ckpt(correlations, d_hidden - 1)

    if ckpt_path and os.path.exists(ckpt_path):
        os.remove(ckpt_path)
        print("Checkpoint removed — pointbiserial complete.")

    return pd.DataFrame(correlations, columns=["neuron_id", "r", "p_value"])

In [20]:
import json
OUTPUT_DIR = f"/content/drive/MyDrive/{model_name}_Activations/Sycophancy_rankings"
os.makedirs(OUTPUT_DIR, exist_ok=True)
BASE_DIR = f"/content/drive/MyDrive/{model_name}_Activations/SAES"
import gc
# Separate checkpoint dirs for each analysis type
CKPT_DIR_RH   = "/content/drive/MyDrive/correlation_checkpoints/reward_hacking"
CKPT_DIR_SYCO = "/content/drive/MyDrive/correlation_checkpoints/sycophancy"
os.makedirs(CKPT_DIR_RH,   exist_ok=True)
os.makedirs(CKPT_DIR_SYCO, exist_ok=True)

# Build chunk paths once before the loop
hacked_chunks      = sorted([
    os.path.join(base_dir, "X_hacked_chunks", f)
    for f in os.listdir(os.path.join(base_dir, "X_hacked_chunks"))
    if f.startswith("chunk_")
])
clean_chunks       = sorted([
    os.path.join(base_dir, "X_clean_chunks", f)
    for f in os.listdir(os.path.join(base_dir, "X_clean_chunks"))
    if f.startswith("chunk_")
])
hacked_syco_chunks = sorted([
    os.path.join(base_dir, "X_hacked_syco_chunks", f)
    for f in os.listdir(os.path.join(base_dir, "X_hacked_syco_chunks"))
    if f.startswith("chunk_")
])
clean_syco_chunks  = sorted([
    os.path.join(base_dir, "X_clean_syco_chunks", f)
    for f in os.listdir(os.path.join(base_dir, "X_clean_syco_chunks"))
    if f.startswith("chunk_")
])

print(f"RB hacked chunks:   {len(hacked_chunks)}")
print(f"RB clean chunks:    {len(clean_chunks)}")
print(f"Syco hacked chunks: {len(hacked_syco_chunks)}")
print(f"Syco clean chunks:  {len(clean_syco_chunks)}")

all_rankings = []

for i in range(1, 4):
    print(f"\n{'='*50}")
    print(f"Processing SAE {i}...")

    sae_path = os.path.join(BASE_DIR, f"matryoshka_sae_{i}.pt")
    sae = torch.load(sae_path, map_location="cuda", weights_only=False)
    sae.eval()
    sae.cuda()
    sae.float()

    # ── Reward hacking analysis ────────────────────────────────────────
    print(f"\n  Reward hacking analysis...")
    df_rank = rank_sae_neurons_chunked(
        sae, hacked_chunks, clean_chunks,
        sae_level=i,
        checkpoint_dir=CKPT_DIR_RH,
        checkpoint_interval=100
    )
    pb_df = compute_pointbiserial(
        hacked_chunks, clean_chunks,
        sae=sae,
        checkpoint_dir=CKPT_DIR_RH,
        checkpoint_interval=500
    )

    path_rh_rank = os.path.join(OUTPUT_DIR, f"sae_normal_{i}_neuron_ranking.csv")
    path_rh_pb   = os.path.join(OUTPUT_DIR, f"sae_point_biserial_{i}_neuron_ranking.csv")
    df_rank.to_csv(path_rh_rank, index=False)
    pb_df.to_csv(path_rh_pb,     index=False)
    print(f"  Saved → {path_rh_rank}")
    print(f"  Saved → {path_rh_pb}")

    # ── Sycophancy analysis ────────────────────────────────────────────
    print(f"\n  Sycophancy analysis...")
    df_rank_syco = rank_sae_neurons_chunked(
        sae, hacked_syco_chunks, clean_syco_chunks,
        sae_level=i,
        checkpoint_dir=CKPT_DIR_SYCO,
        checkpoint_interval=100
    )
    pb_syco = compute_pointbiserial(
        hacked_syco_chunks, clean_syco_chunks,
        sae=sae,
        checkpoint_dir=CKPT_DIR_SYCO,
        checkpoint_interval=500
    )

    path_syco_rank = os.path.join(OUTPUT_DIR, f"sae_normal_syco_{i}_neuron_ranking.csv")
    path_syco_pb   = os.path.join(OUTPUT_DIR, f"sae_point_biserial_syco_{i}_neuron_ranking.csv")
    df_rank_syco.to_csv(path_syco_rank, index=False)
    pb_syco.to_csv(path_syco_pb,        index=False)
    print(f"  Saved → {path_syco_rank}")
    print(f"  Saved → {path_syco_pb}")

    display(df_rank.head(10))
    all_rankings.append(df_rank)

    del sae
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Unloaded SAE {i}.")

print("\nAll SAEs processed.")

RB hacked chunks:   1
RB clean chunks:    1
Syco hacked chunks: 1
Syco clean chunks:  2

Processing SAE 1...

  Reward hacking analysis...


Clean chunks (SAE 1): 100%|██████████| 1/1 [00:56<00:00, 56.99s/it]


Computing streaming accumulators...


Clean chunks: 100%|██████████| 1/1 [00:43<00:00, 43.62s/it]


Computing correlations from accumulators...
Checkpoint removed — pointbiserial complete.
  Saved → /content/drive/MyDrive/google/gemma-2-2b-it_Activations/Sycophancy_rankings/sae_normal_1_neuron_ranking.csv
  Saved → /content/drive/MyDrive/google/gemma-2-2b-it_Activations/Sycophancy_rankings/sae_point_biserial_1_neuron_ranking.csv

  Sycophancy analysis...


...

Computing streaming accumulators...


Clean chunks: 100%|██████████| 2/2 [03:58<00:00, 119.36s/it]


Computing correlations from accumulators...
Checkpoint removed — pointbiserial complete.
  Saved → /content/drive/MyDrive/google/gemma-2-2b-it_Activations/Sycophancy_rankings/sae_normal_syco_3_neuron_ranking.csv
  Saved → /content/drive/MyDrive/google/gemma-2-2b-it_Activations/Sycophancy_rankings/sae_point_biserial_syco_3_neuron_ranking.csv


,sae_level,neuron_id,mean_hacked,mean_clean,freq_hacked,freq_clean,variance_hacked,delta,freq_diff,score
30963,3,30963,0.006638,0.004972,0.347683,0.274774,0.000163,0.001666,0.072909,0.009514
35124,3,35124,0.011635,0.009543,0.555061,0.485918,0.000235,0.002091,0.069143,0.009432
64116,3,64116,0.011650,0.009599,0.520799,0.454018,0.000300,0.002052,0.066781,0.007912
67737,3,67737,0.009180,0.007621,0.519750,0.454216,0.000170,0.001559,0.065534,0.007848
63694,3,63694,0.018029,0.015087,0.623698,0.569488,0.000467,0.002942,0.054210,0.007379
24138,3,24138,0.015050,0.012855,0.668700,0.612379,0.000287,0.002195,0.056321,0.007298
49781,3,49781,0.006922,0.005399,0.371485,0.311434,0.000159,0.001524,0.060051,0.007263
58352,3,58352,0.014963,0.012697,0.659140,0.606549,0.000272,0.002266,0.052591,0.007227
20277,3,20277,0.013415,0.011384,0.634438,0.579234,0.000247,0.002031,0.055203,0.007131
33549,3,33549,0.009394,0.007801,0.496834,0.436334,0.000185,0.001593,0.060499,0.007086


Unloaded SAE 3.

All SAEs processed.


In [21]:
print(type(hacked_chunks[0]))
print(hacked_chunks[0])

<class 'str'>
/content/drive/MyDrive/X_hacked_chunks/chunk_0000.pt


In [22]:
print('Re-checking GPU memory usage:')
!nvidia-smi

Re-checking GPU memory usage:
Wed Apr 22 03:29:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             49W /  400W |     920MiB /  40960MiB |     22%      Default |
|                                         |                        |             Disabled |
+-----------------

In [23]:
print('Checking GPU memory usage after restart:')
!nvidia-smi

Checking GPU memory usage after restart:
Wed Apr 22 03:29:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             63W /  400W |     920MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+------

In [24]:
print('Checking GPU memory usage:')
!nvidia-smi

Checking GPU memory usage:
Wed Apr 22 03:29:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             68W /  400W |     920MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+--------------------

In [25]:
# Clear previously loaded SAEs and free memory
if 'saes' in globals():
    del saes
torch.cuda.empty_cache()
print("Cleared 'saes' variable and CUDA cache.")

Cleared 'saes' variable and CUDA cache.


In [26]:
print(len(saes))

NameError: name 'saes' is not defined